# Fashion MNIST Deep Learning Project
## Сучасні технології Deep Learning

Це завдання реалізує повний аналіз класифікації Fashion MNIST з використанням нейронних мереж, включаючи оптимізацію гіперпараметрів та детальний аналіз результатів.

In [ ]:
# Імпорт необхідних бібліотек
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import utils
from PIL import Image
import time
from datetime import datetime
import os

# Налаштування для відтворюваності
tf.random.set_seed(42)
np.random.seed(42)

print("Версія TensorFlow:", tf.__version__)

## 1. Завантаження та підготовка даних

In [ ]:
# Завантажуємо дані
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Список з назвами класів
classes = ['футболка', 'штани', 'светр', 'плаття', 'пальто', 
           'туфлі', 'сорочка', 'кросівки', 'сумка', 'черевики']

print(f"Розмір тренувальних даних: {x_train.shape}")
print(f"Розмір тестових даних: {x_test.shape}")
print(f"Кількість класів: {len(classes)}")

# Візуалізація прикладів даних
plt.figure(figsize=(12, 8))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f'{classes[y_train[i]]}')
    plt.axis('off')
plt.suptitle('Приклади з Fashion MNIST')
plt.tight_layout()
plt.show()

In [ ]:
# Перетворення розмірності зображень
x_train_flat = x_train.reshape(60000, 784)
x_test_flat = x_test.reshape(10000, 784)

# Нормалізація даних
x_train_flat = x_train_flat / 255.0
x_test_flat = x_test_flat / 255.0

# Перетворимо мітки в категорії
y_train_cat = utils.to_categorical(y_train, 10)
y_test_cat = utils.to_categorical(y_test, 10)

print(f"Форма плоских тренувальних даних: {x_train_flat.shape}")
print(f"Форма категорійних міток: {y_train_cat.shape}")

## 2. Базова модель

In [ ]:
# Створюємо послідовну модель
model = Sequential()

# Додаємо рівні мережі
model.add(Dense(800, input_dim=784, activation="relu"))
model.add(Dense(10, activation="softmax"))

# Компілюємо модель
model.compile(loss="categorical_crossentropy", 
              optimizer="SGD", 
              metrics=["accuracy"])

print(model.summary())

In [ ]:
# Навчаємо мережу
start_time = time.time()

history = model.fit(x_train_flat, y_train_cat, 
                    batch_size=200, 
                    epochs=100,
                    validation_split=0.2,
                    verbose=1)

training_time = time.time() - start_time
print(f"\nЧас навчання: {training_time:.2f} секунд")

In [ ]:
# Оцінюємо якість навчання мережі на тестових даних
scores = model.evaluate(x_test_flat, y_test_cat, verbose=1)
base_accuracy = round(scores[1] * 100, 4)
print(f"\nЧастка вірних відповідей на тестових даних, у відсотках: {base_accuracy}")

In [ ]:
# Візуалізація процесу навчання
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Точність моделі')
plt.xlabel('Епохи')
plt.ylabel('Точність')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Функція втрат')
plt.xlabel('Епохи')
plt.ylabel('Втрати')
plt.legend()

plt.tight_layout()
plt.show()

## 3. Розпізнавання зображення з набору даних

In [ ]:
# Розпізнавання конкретного зображення
n_rec = 492  # індекс зображення (сумка)

# Отримуємо зображення та справжню мітку
test_image = x_test_flat[n_rec].reshape(1, 784)
true_label = y_test[n_rec]
true_class = classes[true_label]

# Робимо передбачення
prediction = model.predict(test_image)
predicted_label = np.argmax(prediction)
predicted_class = classes[predicted_label]
confidence = prediction[0][predicted_label] * 100

print(f"Зображення номер {n_rec}")
print(f"Справжній клас: {true_class} (індекс {true_label})")
print(f"Передбачений клас: {predicted_class} (індекс {predicted_label})")
print(f"Впевненість: {confidence:.2f}%")
print(f"Розпізнавання {'ПРАВИЛЬНЕ' if true_label == predicted_label else 'НЕПРАВИЛЬНЕ'}")

In [ ]:
# Візуалізація результату розпізнавання
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(x_test[n_rec], cmap='gray')
plt.title(f"Зображення {n_rec}\nСправжній: {true_class}")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.bar(range(10), prediction[0])
plt.xticks(range(10), [cls[:8] for cls in classes], rotation=45)
plt.title(f"Розподіл ймовірностей\nПередбачення: {predicted_class}")
plt.ylabel('Ймовірність')

# Показуємо топ-3 передбачення
top3_indices = np.argsort(prediction[0])[-3:][::-1]
plt.subplot(1, 3, 3)
top3_probs = [prediction[0][i] for i in top3_indices]
top3_classes = [classes[i] for i in top3_indices]
plt.barh(range(3), top3_probs)
plt.yticks(range(3), [f"{cls[:10]}" for cls in top3_classes])
plt.title("Топ-3 передбачення")
plt.xlabel('Ймовірність')

plt.tight_layout()
plt.show()

## 4. Створення та розпізнавання власних зображень

In [ ]:
# Створення простого зразка футболки
def create_sample_tshirt():
    tshirt = np.zeros((28, 28))
    # Контур футболки
    tshirt[8:12, 6:22] = 255  # лінія плечей
    tshirt[12:24, 10:18] = 255  # тіло
    tshirt[6:10, 8:20] = 255   # область шиї
    return tshirt

# Створення простого зразка взуття
def create_sample_shoe():
    shoe = np.zeros((28, 28))
    # Контур взуття
    shoe[16:24, 4:24] = 255  # підошва
    shoe[12:20, 6:22] = 255  # верхня частина
    shoe[14:16, 8:20] = 255  # область шнурків
    return shoe

# Створюємо зразки
sample_tshirt = create_sample_tshirt()
sample_shoe = create_sample_shoe()

# Візуалізація створених зображень
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_tshirt, cmap='gray')
plt.title('Зразок футболки')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(sample_shoe, cmap='gray')
plt.title('Зразок взуття')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Функція для розпізнавання власного зображення
def predict_custom_image(img_array, title="Власне зображення"):
    # Нормалізація та перетворення для моделі
    img_normalized = img_array / 255.0
    img_flat = img_normalized.reshape(1, 784)
    
    # Робимо передбачення
    prediction = model.predict(img_flat)
    predicted_label = np.argmax(prediction)
    predicted_class = classes[predicted_label]
    confidence = prediction[0][predicted_label] * 100
    
    print(f"\n{title}:")
    print(f"Передбачений клас: {predicted_class} (індекс {predicted_label})")
    print(f"Впевненість: {confidence:.2f}%")
    
    # Показуємо топ-3 передбачення
    top3_indices = np.argsort(prediction[0])[-3:][::-1]
    print("\nТоп-3 передбачення:")
    for i, idx in enumerate(top3_indices, 1):
        print(f"{i}. {classes[idx]}: {prediction[0][idx]*100:.2f}%")
    
    # Візуалізація
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img_array, cmap='gray')
    plt.title(f"{title}\nПередбачення: {predicted_class}")
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.bar(range(10), prediction[0])
    plt.xticks(range(10), [cls[:8] for cls in classes], rotation=45)
    plt.title(f"Розподіл ймовірностей")
    plt.ylabel('Ймовірність')
    
    plt.tight_layout()
    plt.show()
    
    return predicted_label, confidence

# Тестуємо наші зразки
predict_custom_image(sample_tshirt, "Зразок футболки")
predict_custom_image(sample_shoe, "Зразок взуття")

## 5. Експерименти з гіперпараметрами

### 5.1 Експеримент з кількістю епох

In [ ]:
def experiment_epochs(epochs_list=[50, 75, 100, 125]):
    """Експеримент з різною кількістю епох"""
    print("=== ЕКСПЕРИМЕНТ: КІЛЬКІСТЬ ЕПОХ ===")
    print(f"Тестуємо епохи: {epochs_list}")
    
    results = {}
    
    for epochs in epochs_list:
        print(f"\nНавчання з {epochs} епохами...")
        
        # Створюємо нову модель
        temp_model = Sequential()
        temp_model.add(Dense(800, input_dim=784, activation="relu"))
        temp_model.add(Dense(10, activation="softmax"))
        temp_model.compile(loss="categorical_crossentropy", optimizer="SGD", metrics=["accuracy"])
        
        # Навчання
        start_time = time.time()
        history = temp_model.fit(x_train_flat, y_train_cat, 
                                batch_size=200, epochs=epochs,
                                validation_split=0.2, verbose=0)
        training_time = time.time() - start_time
        
        # Оцінка
        scores = temp_model.evaluate(x_test_flat, y_test_cat, verbose=0)
        accuracy = round(scores[1] * 100, 4)
        
        results[epochs] = {
            'accuracy': accuracy,
            'training_time': training_time,
            'history': history
        }
        
        print(f"Епохи: {epochs}, Точність: {accuracy}%, Час: {training_time:.1f}с")
    
    # Визначаємо найкращий результат
    best_epochs = max(results.keys(), key=lambda k: results[k]['accuracy'])
    print(f"\nНайкращий результат: {best_epochs} епох з точністю {results[best_epochs]['accuracy']}%")
    
    return results

# Запускаємо експеримент
epochs_results = experiment_epochs()

In [ ]:
# Візуалізація результатів експерименту з епохами
epochs_list = list(epochs_results.keys())
accuracies = [epochs_results[e]['accuracy'] for e in epochs_list]
times = [epochs_results[e]['training_time'] for e in epochs_list]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_list, accuracies, 'bo-')
plt.title('Точність залежно від кількості епох')
plt.xlabel('Кількість епох')
plt.ylabel('Точність (%)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_list, times, 'ro-')
plt.title('Час навчання залежно від кількості епох')
plt.xlabel('Кількість епох')
plt.ylabel('Час (секунди)')
plt.grid(True)

plt.tight_layout()
plt.show()

### 5.2 Експеримент з розміром міні-вибірки

In [ ]:
def experiment_batch_size(batch_sizes=[50, 100, 200, 400]):
    """Експеримент з різними розмірами batch"""
    print("=== ЕКСПЕРИМЕНТ: РОЗМІР МІНІ-ВИБІРКИ ===")
    print(f"Тестуємо batch_size: {batch_sizes}")
    
    results = {}
    
    for batch_size in batch_sizes:
        print(f"\nНавчання з batch_size={batch_size}...")
        
        # Створюємо нову модель
        temp_model = Sequential()
        temp_model.add(Dense(800, input_dim=784, activation="relu"))
        temp_model.add(Dense(10, activation="softmax"))
        temp_model.compile(loss="categorical_crossentropy", optimizer="SGD", metrics=["accuracy"])
        
        # Навчання
        start_time = time.time()
        history = temp_model.fit(x_train_flat, y_train_cat, 
                                batch_size=batch_size, epochs=100,
                                validation_split=0.2, verbose=0)
        training_time = time.time() - start_time
        
        # Оцінка
        scores = temp_model.evaluate(x_test_flat, y_test_cat, verbose=0)
        accuracy = round(scores[1] * 100, 4)
        
        results[batch_size] = {
            'accuracy': accuracy,
            'training_time': training_time,
            'history': history
        }
        
        print(f"Batch size: {batch_size}, Точність: {accuracy}%, Час: {training_time:.1f}с")
    
    # Визначаємо найкращий результат
    best_batch_size = max(results.keys(), key=lambda k: results[k]['accuracy'])
    print(f"\nНайкращий результат: batch_size={best_batch_size} з точністю {results[best_batch_size]['accuracy']}%")
    
    return results

# Запускаємо експеримент
batch_results = experiment_batch_size()

### 5.3 Експеримент з кількістю нейронів

In [ ]:
def experiment_hidden_neurons(neuron_counts=[500, 700, 900, 1200]):
    """Експеримент з різною кількістю нейронів у прихованому шарі"""
    print("=== ЕКСПЕРИМЕНТ: КІЛЬКІСТЬ НЕЙРОНІВ ВХІДНОГО ШАРУ ===")
    print(f"Тестуємо кількість нейронів: {neuron_counts}")
    
    results = {}
    
    for neurons in neuron_counts:
        print(f"\nНавчання з {neurons} нейронами...")
        
        # Створюємо модель з різною кількістю нейронів
        temp_model = Sequential()
        temp_model.add(Dense(neurons, input_dim=784, activation="relu"))
        temp_model.add(Dense(10, activation="softmax"))
        temp_model.compile(loss="categorical_crossentropy", optimizer="SGD", metrics=["accuracy"])
        
        # Навчання
        start_time = time.time()
        history = temp_model.fit(x_train_flat, y_train_cat, 
                                batch_size=200, epochs=100,
                                validation_split=0.2, verbose=0)
        training_time = time.time() - start_time
        
        # Оцінка
        scores = temp_model.evaluate(x_test_flat, y_test_cat, verbose=0)
        accuracy = round(scores[1] * 100, 4)
        
        results[neurons] = {
            'accuracy': accuracy,
            'training_time': training_time,
            'history': history
        }
        
        print(f"Нейронів: {neurons}, Точність: {accuracy}%, Час: {training_time:.1f}с")
    
    # Визначаємо найкращий результат
    best_neurons = max(results.keys(), key=lambda k: results[k]['accuracy'])
    print(f"\nНайкращий результат: {best_neurons} нейронів з точністю {results[best_neurons]['accuracy']}%")
    
    return results

# Запускаємо експеримент
neurons_results = experiment_hidden_neurons()

### 5.4 Експеримент з додатковим прихованим шаром

In [ ]:
def experiment_additional_layer(layer_neurons=[500, 700, 900, 1200]):
    """Експеримент з додатковим прихованим шаром"""
    print("=== ЕКСПЕРИМЕНТ: ДОДАТКОВІ ПРИХОВАНІ ШАРИ ===")
    print(f"Тестуємо додаткові шари з нейронами: {layer_neurons}")
    
    results = {}
    
    for neurons in layer_neurons:
        print(f"\nНавчання з додатковим шаром ({neurons} нейронів)...")
        
        # Створюємо модель з додатковим прихованим шаром
        temp_model = Sequential()
        temp_model.add(Dense(800, input_dim=784, activation="relu"))
        temp_model.add(Dense(neurons, activation="relu"))  # Додатковий прихований шар
        temp_model.add(Dense(10, activation="softmax"))
        temp_model.compile(loss="categorical_crossentropy", optimizer="SGD", metrics=["accuracy"])
        
        # Навчання
        start_time = time.time()
        history = temp_model.fit(x_train_flat, y_train_cat, 
                                batch_size=200, epochs=100,
                                validation_split=0.2, verbose=0)
        training_time = time.time() - start_time
        
        # Оцінка
        scores = temp_model.evaluate(x_test_flat, y_test_cat, verbose=0)
        accuracy = round(scores[1] * 100, 4)
        
        results[neurons] = {
            'accuracy': accuracy,
            'training_time': training_time,
            'history': history
        }
        
        print(f"Додатковий шар: {neurons} нейронів, Точність: {accuracy}%, Час: {training_time:.1f}с")
    
    # Визначаємо найкращий результат
    best_layer = max(results.keys(), key=lambda k: results[k]['accuracy'])
    print(f"\nНайкращий результат: додатковий шар з {best_layer} нейронів, точність {results[best_layer]['accuracy']}%")
    
    return results

# Запускаємо експеримент
layer_results = experiment_additional_layer()

## 6. Порівняння всіх експериментів

In [ ]:
# Візуалізація всіх результатів експериментів
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Епохи
epochs_list = list(epochs_results.keys())
epochs_acc = [epochs_results[e]['accuracy'] for e in epochs_list]
axes[0,0].plot(epochs_list, epochs_acc, 'bo-')
axes[0,0].set_title('Точність vs Кількість епох')
axes[0,0].set_xlabel('Епохи')
axes[0,0].set_ylabel('Точність (%)')
axes[0,0].grid(True)

# Batch size
batch_list = list(batch_results.keys())
batch_acc = [batch_results[b]['accuracy'] for b in batch_list]
axes[0,1].plot(batch_list, batch_acc, 'ro-')
axes[0,1].set_title('Точність vs Batch Size')
axes[0,1].set_xlabel('Batch Size')
axes[0,1].set_ylabel('Точність (%)')
axes[0,1].grid(True)

# Кількість нейронів
neurons_list = list(neurons_results.keys())
neurons_acc = [neurons_results[n]['accuracy'] for n in neurons_list]
axes[1,0].plot(neurons_list, neurons_acc, 'go-')
axes[1,0].set_title('Точність vs Кількість нейронів')
axes[1,0].set_xlabel('Нейрони')
axes[1,0].set_ylabel('Точність (%)')
axes[1,0].grid(True)

# Додатковий шар
layer_list = list(layer_results.keys())
layer_acc = [layer_results[l]['accuracy'] for l in layer_list]
axes[1,1].plot(layer_list, layer_acc, 'mo-')
axes[1,1].set_title('Точність vs Додатковий шар')
axes[1,1].set_xlabel('Нейрони в додатковому шарі')
axes[1,1].set_ylabel('Точність (%)')
axes[1,1].grid(True)

plt.tight_layout()
plt.show()

## 7. Створення оптимальної моделі

In [ ]:
# Визначення найкращих параметрів
best_epochs = max(epochs_results.keys(), key=lambda k: epochs_results[k]['accuracy'])
best_batch_size = max(batch_results.keys(), key=lambda k: batch_results[k]['accuracy'])
best_neurons = max(neurons_results.keys(), key=lambda k: neurons_results[k]['accuracy'])
best_additional = max(layer_results.keys(), key=lambda k: layer_results[k]['accuracy'])

print("ОПТИМАЛЬНІ ПАРАМЕТРИ:")
print(f"- Епохи: {best_epochs}")
print(f"- Batch size: {best_batch_size}")
print(f"- Нейронів у першому шарі: {best_neurons}")
print(f"- Нейронів у додатковому шарі: {best_additional}")

# Створення оптимальної моделі
optimal_model = Sequential()
optimal_model.add(Dense(best_neurons, input_dim=784, activation="relu"))
optimal_model.add(Dense(best_additional, activation="relu"))
optimal_model.add(Dense(10, activation="softmax"))

optimal_model.compile(loss="categorical_crossentropy", 
                     optimizer="SGD", 
                     metrics=["accuracy"])

print(f"\nАрхітектура оптимальної моделі:")
print(optimal_model.summary())

In [ ]:
# Навчання оптимальної моделі
print("Навчання оптимальної моделі...")
start_time = time.time()

optimal_history = optimal_model.fit(x_train_flat, y_train_cat, 
                                   batch_size=best_batch_size, 
                                   epochs=best_epochs,
                                   validation_split=0.2,
                                   verbose=1)

optimal_training_time = time.time() - start_time

# Оцінка оптимальної моделі
optimal_scores = optimal_model.evaluate(x_test_flat, y_test_cat, verbose=1)
optimal_accuracy = round(optimal_scores[1] * 100, 4)

print(f"\nРЕЗУЛЬТАТИ ОПТИМАЛЬНОЇ МОДЕЛІ:")
print(f"Точність на тестових даних: {optimal_accuracy}%")
print(f"Час навчання: {optimal_training_time:.2f} секунд")
print(f"Поліпшення порівняно з базовою моделлю: {optimal_accuracy - base_accuracy:.4f}%")

## 8. Підсумковий звіт

In [ ]:
# Генерація підсумкового звіту
print("\n" + "="*60)
print("ПІДСУМКОВИЙ ЗВІТ - FASHION MNIST DEEP LEARNING")
print(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

print(f"\n1. БАЗОВА МОДЕЛЬ:")
print(f"   Точність: {base_accuracy}%")

print(f"\n2. РЕЗУЛЬТАТИ ЕКСПЕРИМЕНТІВ:")
print(f"   Кращі епохи: {best_epochs} ({epochs_results[best_epochs]['accuracy']}%)")
print(f"   Кращий batch size: {best_batch_size} ({batch_results[best_batch_size]['accuracy']}%)")
print(f"   Кращі нейрони: {best_neurons} ({neurons_results[best_neurons]['accuracy']}%)")
print(f"   Кращий додатковий шар: {best_additional} ({layer_results[best_additional]['accuracy']}%)")

print(f"\n3. ОПТИМАЛЬНА МОДЕЛЬ:")
print(f"   Точність: {optimal_accuracy}%")
print(f"   Поліпшення: +{optimal_accuracy - base_accuracy:.4f}%")

print(f"\n4. ВИСНОВКИ:")
if optimal_accuracy > base_accuracy:
    print(f"   ✓ Оптимізація гіперпараметрів покращила результат")
else:
    print(f"   ⚠ Оптимізація не дала значного покращення")

print(f"\n5. РЕКОМЕНДАЦІЇ ДЛЯ ПОДАЛЬШОГО ПОКРАЩЕННЯ:")
print(f"   - Використання CNN архітектури")
print(f"   - Застосування Dropout для регуляризації")
print(f"   - Використання Adam або RMSprop оптимізаторів")
print(f"   - Аугментація даних")
print(f"   - Тонке налаштування learning rate")

print("\n" + "="*60)
print("ПРОЕКТ ЗАВЕРШЕНО УСПІШНО!")
print("="*60)